# 30 — ResNet18 + Grad-CAM (Quỳnh) — KHUNG

**Việc của Quỳnh trước khi chạy:**
1. Cài đặt `fit / predict_proba / save / load` trong `src/banana_ripeness/models/resnet18.py` (`TODO(Quỳnh)`).
2. Cài đặt Grad-CAM trong `src/banana_ripeness/models/gradcam.py`.
3. Khai báo thí nghiệm trong `configs/experiments.yaml` (mục `quynh`; đang để 3 seed để tính std).
4. Push code lên GitHub, rồi chạy notebook này trên Kaggle (**Accelerator: GPU**, Internet On).

Khung đã có: đọc `stats.json` (`train_mean_std()`, `train_class_weight()`), `load_image(...)`, chia dữ liệu,
độ đo, bootstrap std, std giữa các seed, lưu kết quả cùng định dạng để notebook 90 gộp với SVM / HOG.

In [ ]:
# === SETUP: chạy cell này đầu tiên (Kaggle / Colab / máy) ===
# Kaggle: bật Settings -> Internet = On. Code lấy từ GitHub, dữ liệu lấy từ Kaggle Dataset (Add Input).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/LuanLuKim/cs231-banana-ripeness.git"
BRANCH = "main"
OWNER = "quynh"          # tên người chạy, ghi vào bảng kết quả chung

local = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "banana_ripeness").exists()), None)
if local is not None:                       # chạy trên máy, trong thư mục repo
    CODE_DIR = local
else:                                       # Kaggle / Colab: clone vào /tmp (không lẫn vào output notebook)
    CODE_DIR = Path("/tmp/cs231-banana-ripeness")
    if (CODE_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "-q", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE_DIR)], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
os.environ["BANANA_OWNER"] = OWNER
commit = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code:", CODE_DIR, "| commit", commit or "?")

%load_ext autoreload
%autoreload 2

In [ ]:
from banana_ripeness.data import find_data_root, load_manifest, get_split
import pandas as pd
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)

# Tự tìm thư mục có manifest.csv trong /kaggle/input (Kaggle Dataset output preprocessing đã Add Input).
# Nếu có nhiều bản, chỉ rõ: DATA_ROOT = find_data_root("/kaggle/input/<ten-dataset>")
DATA_ROOT = find_data_root()
df = load_manifest(DATA_ROOT)
EXP = CODE_DIR / "configs" / "experiments.yaml"
print("Dữ liệu:", DATA_ROOT, "|", len(df), "ảnh")
pd.crosstab([df.split_mixed], df.label).loc[["train", "val", "test"]]

In [ ]:
from banana_ripeness.experiment import load_experiments, run_grid, load_results, compare_table
res = run_grid(load_experiments(EXP, "quynh"), df=df, data_root=DATA_ROOT, owner=OWNER)
res = load_results(); res = res[res.owner == OWNER]
display(compare_table(res, split="test", scheme="mixed"))        # std giữa các seed nếu > 1 seed
display(compare_table(res, split="external", scheme="external"))

In [ ]:
from banana_ripeness.experiment import best_run
from banana_ripeness.models import load_model
best = best_run(res, scheme="mixed", by_split="val")
best.plot_confusion("test");
model = load_model(best.run_dir)

In [ ]:
# Grad-CAM (TODO Quỳnh: cài đặt gradcam_grid)
from banana_ripeness.models.gradcam import gradcam_grid
test = get_split(df, "mixed", "test")
gradcam_grid(model, test.groupby("label").sample(2, random_state=42));